# Executor parity fixture

Deterministic exercise of the pinned scientific stack — NO network calls,
fixed inputs, seeded RNG. Executed by `scripts/executor-parity.mjs` on each
notebook-executor driver; normalized results must match across drivers.

In [ ]:
import logging
logging.getLogger('matplotlib').setLevel(logging.ERROR)  # suppress font-cache notice
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
print('numpy', np.__version__)
print('pandas', pd.__version__)
print('matplotlib', matplotlib.__version__)

In [ ]:
rng = np.random.default_rng(20260802)
values = rng.integers(0, 100, size=12)
print(values.tolist())
print(int(values.sum()), float(values.mean()))

In [ ]:
df = pd.DataFrame({
    'district': ['north', 'south', 'east', 'west'] * 3,
    'requests': values,
})
summary = df.groupby('district', sort=True)['requests'].agg(['count', 'sum', 'mean'])
print(summary.to_string())

In [ ]:
import hashlib, io
fig, ax = plt.subplots(figsize=(4, 3), dpi=100)
summary['sum'].plot.bar(ax=ax, color='#4477AA')
ax.set_title('requests by district (fixture)')
fig.tight_layout()
buf = io.BytesIO()
# metadata={'Software': None} drops the version-bearing PNG text chunk so
# the bytes depend only on the pinned matplotlib wheel's rendering.
fig.savefig(buf, format='png', metadata={'Software': None})
png_bytes = buf.getvalue()
print('png_sha256', hashlib.sha256(png_bytes).hexdigest())
print('png_len', len(png_bytes))

In [ ]:
from IPython.display import Image, display
display(Image(data=png_bytes))
plt.close(fig)